# Atelier Préparation de Données Tabulaires

## Contexte
Une entreprise exploite plusieurs bâtiments intelligents équipés de capteurs IoT. Chaque capteur
collecte régulièrement des informations sur la température, l'humidité, la qualité de l'air, la
consommation énergétique, le nombre de personnes présentes, le type de bâtiment, le mode de
fonctionnement et l'état du système de climatisation.
Les données collectées sont destinées à alimenter ultérieurement un modèle de Machine Learning
capable de prédire la consommation énergétique ou de détecter les situations anormales.
Cependant, les données brutes présentent volontairement différents problèmes : valeurs
manquantes, doublons, valeurs aberrantes, types incorrects, valeurs incohérentes, variables
catégorielles, catégories rares, déséquilibre des classes et échelles différentes entre variables.
L'objectif de l'atelier est donc de transformer le fichier brut en un jeu de données propre et prêt
pour le Machine Learning.

## Objectifs pédagogiques
À la fin de l'atelier, l'apprenant doit être capable de :
1) explorer un jeu de données tabulaire ;
2) identifier les différents types de variables ;
3) détecter les problèmes de qualité ;
4) analyser les valeurs manquantes ;
5) détecter et traiter les doublons ;
6) identifier les valeurs aberrantes ;
7) détecter les incohérences ;
8) analyser le déséquilibre des classes ;
9) choisir une stratégie d'encodage adaptée ;
10) appliquer différents encodages catégoriels ;
11) normaliser et standardiser les variables numériques ;
12) éviter la fuite de données ;
13) construire un pipeline de preprocessing avec Scikit-learn ;
14) produire un dataset final exploitable par un algorithme de Machine Learning.

### Partie 1 – Explorer les données

### 1) Charger les données CSV

In [1]:
pip install pandas

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.


In [2]:
import pandas as pd

df = pd.read_csv("../data/smart_building_raw.csv")

### 2) Afficher les premières lignes du dataset

In [3]:
df.head()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
0,1174,2025-02-13 06:00:00,B8,Entrepôt,A,23.0,40.0,851.0,26.0,138.4,Eco,Normal,Jeudi,Non
1,1275,2025-03-10 12:00:00,B7,Bureau,D,28.0,69.0,538.0,0.0,170.2,Eco,Normal,Lundi,Non
2,1493,2025-05-04 00:00:00,B5,Centre commercial,B,19.5,52.3,1198.0,64.0,149.8,Normal,Normal,Dimanche,Oui
3,1073,2025-01-19 00:00:00,B6,Université,D,20.0,58.4,1014.0,30.0,141.6,Normal,Normal,Dimanche,Non
4,1454,2025-04-24 06:00:00,B7,Bureau,D,26.8,70.5,628.0,42.0,228.3,Normal,Normal,Jeudi,Non


### 3) Afficher les dernières lignes du dataset

In [4]:
df.tail()

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
502,1107,2025-01-27 12:00:00,B2,École,C,29.8,66.5,778.0,48.0,243.5,Normal,Normal,Lundi,Non
503,1271,2025-03-09 12:00:00,B3,Hôpital,B,21.1,66.3,3900.0,53.0,145.8,Eco,Alerte,Dimanche,Oui
504,1349,2025-03-29 00:00:00,B5,Centre commercial,B,21.6,61.9,959.0,33.0,100.2,Eco,Normal,Samedi,Non
505,1436,2025-04-19 18:00:00,B7,Bureau,C,21.2,54.5,1066.0,38.0,150.9,Boost,Normal,Samedi,Oui
506,1103,2025-01-26 12:00:00,B2,École,D,NaN,64.8,584.0,29.0,163.4,Normal,Panne,Dimanche,Non


### 4) Combien d'observations contient le dataset

In [5]:
df.shape[0]

507

### 5) Combien de variables possède le dataset

In [6]:
df.shape[1]

14

### 6) Identifier les variables numériques

In [7]:
df.select_dtypes(include="number").columns

Index(['id_mesure', 'temperature', 'humidite', 'co2', 'occupation',
       'consommation_kwh'],
      dtype='str')

### 7) Identifier les variables catégorielles

In [8]:
df.select_dtypes(include="object").columns

/tmp/ipykernel_5220/1196084386.py:1: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  df.select_dtypes(include="object").columns


Index(['date', 'batiment', 'type_batiment', 'zone', 'mode_climatisation',
       'etat_systeme', 'jour_semaine', 'alerte'],
      dtype='str')

### 8) Identifier les dates

In [9]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["date"].head()

0   2025-02-13 06:00:00
1   2025-03-10 12:00:00
2   2025-05-04 00:00:00
3   2025-01-19 00:00:00
4   2025-04-24 06:00:00
Name: date, dtype: datetime64[us]

### 9) Identifier les identifiants

In [10]:
df["id_mesure"]

0      1174
1      1275
2      1493
3      1073
4      1454
       ... 
502    1107
503    1271
504    1349
505    1436
506    1103
Name: id_mesure, Length: 507, dtype: int64

### 10) Déterminer les statistiques : moyenne, médiane, minimum, maximum, écart-type et quartiles

In [11]:
df.describe()

,id_mesure,date,temperature,humidite,co2,occupation,consommation_kwh
count,507.000000,502,495.000000,496.000000,500.000000,501.000000,502.000000
mean,1251.114398,2025-03-04 13:26:03.346613,24.154141,57.864113,844.150000,44.850299,169.069323
min,1001.000000,2025-01-01 00:00:00,-30.000000,-12.000000,89.000000,-20.000000,-100.000000
25%,1125.500000,2025-02-01 01:30:00,21.600000,49.275000,623.750000,27.000000,136.875000
50%,1252.000000,2025-03-04 21:00:00,24.000000,57.550000,787.500000,46.000000,169.800000
75%,1376.500000,2025-04-04 22:30:00,26.000000,65.750000,952.000000,61.000000,202.975000
max,1500.000000,2025-05-05 18:00:00,96.000000,160.000000,6000.000000,116.000000,336.200000
std,144.782769,NaN,7.418465,16.026336,582.181386,24.949139,53.164294


### 11) Y a-t-il des variables potentiellement problématiques ?

L'exploration révèle plusieurs variables potentiellement problématiques. Sur le plan des types, id_mesure est stocké comme une variable numérique alors qu'il s'agit en réalité d'un identifiant, sans signification statistique. La colonne date, initialement de type texte, a nécessité une conversion explicite en datetime, et certaines valeurs n'ont pas pu être interprétées. Sur le plan des valeurs, humidite (exprimée en pourcentage) présente potentiellement des valeurs hors de la plage logique [0, 100], tandis que temperature et consommation_kwh affichent des écarts importants entre leurs valeurs minimales et maximales, suggérant la présence de valeurs aberrantes ou de mesures physiquement incohérentes (température extrême, consommation négative). Enfin, les variables catégorielles (type_batiment, mode_climatisation, etat_systeme, jour_semaine, alerte) doivent être examinées pour détecter d'éventuelles catégories mal orthographiées, des espaces parasites, ou une casse incohérente (majuscules/minuscules mélangées), ainsi qu'un possible déséquilibre de classes, en particulier sur la variable cible alerte.

### 12) Pour les données incohérentes :

#### a) rechercher des valeurs telles que humidité < 0

In [12]:
df[df["humidite"] < 0]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
281,1126,2025-02-01 06:00:00,B2,École,D,24.8,-5.0,759.0,57.0,225.2,Boost,Normal,Samedi,Non
335,1036,2025-01-09 18:00:00,B4,Bureau,D,21.2,-8.0,160.0,25.0,120.2,Normal,Alerte,Jeudi,Non
366,1216,2025-02-23 18:00:00,B3,Hôpital,B,23.2,-12.0,702.0,41.0,119.8,Normal,Normal,Dimanche,Non


### b) rechercher des valeurs telles que humidité > 100

In [13]:
df[df["humidite"] > 100]

,id_mesure,date,batiment,type_batiment,zone,temperature,humidite,co2,occupation,consommation_kwh,mode_climatisation,etat_systeme,jour_semaine,alerte
59,1246,2025-03-03 06:00:00,B5,Centre commercial,A,24.2,108.0,813.0,18.0,-15.0,Normal,Normal,Lundi,Non
103,1016,2025-01-04 18:00:00,B6,Université,B,26.9,145.0,670.0,28.0,175.9,Normal,Normal,Samedi,Non
128,1156,2025-02-08 18:00:00,B1,Bureau,A,NaN,160.0,941.0,37.0,149.8,Eco,Normal,Samedi,Oui
160,1186,2025-02-16 06:00:00,B8,Entrepôt,B,24.9,125.0,NaN,18.0,123.5,Normal,Normal,Dimanche,Non
268,1276,2025-03-10 18:00:00,B1,Bureau,D,26.0,140.0,633.0,25.0,148.3,Normal,Normal,Lundi,Non
327,1066,2025-01-17 06:00:00,B2,École,C,25.8,132.0,646.0,14.0,93.6,Eco,Normal,Vendredi,Non
342,1096,2025-01-24 18:00:00,B8,Entrepôt,A,20.3,110.0,783.0,55.0,121.0,Normal,Normal,Vendredi,Non


### c) rechercher des valeurs telles que température extrêmement élevée

In [14]:
seuil_max = 45   # °C, seuil choisi pour un bâtiment

temp_elevee = df[df["temperature"] > seuil_max]
print("Nombre de températures > 45 °C :", len(temp_elevee))
temp_elevee[["id_mesure", "date", "batiment", "temperature"]]

Nombre de températures > 45 °C : 4


,id_mesure,date,batiment,temperature
7,1141,2025-02-05,B2,72.5
116,1181,2025-02-15,B6,96.0
161,1061,2025-01-16,B5,88.0
499,1021,2025-01-06,B2,95.2


Seuil retenu : 45 °C. La plus haute température plausible du jeu est 34,2 °C, et l'on passe directement à 72,5 °C, ce qui laisse un écart net. Pour un bâtiment, une température intérieure de 72 à 96 °C est physiquement impossible (incendie exclu, elle ne serait pas mesurée par un capteur de confort). Ces 4 valeurs sont donc des erreurs de capteur. Elles seront converties en NaN en 12f, puisque leur vraie valeur est introuvable.

### d) rechercher des valeurs telles que occupation négative

In [15]:
occ_negative = df[df["occupation"] < 0]

print("Nombre d'occupations négatives :", len(occ_negative))
occ_negative[["id_mesure", "date", "batiment", "occupation"]]

Nombre d'occupations négatives : 5


,id_mesure,date,batiment,occupation
22,1031,2025-01-08 12:00:00,B2,-5.0
376,1231,2025-02-27 12:00:00,B7,-8.0
430,1081,2025-01-21 00:00:00,B1,-12.0
487,1131,2025-02-02 12:00:00,B1,-2.0
494,1331,2025-03-24 12:00:00,B5,-20.0


### e) rechercher des valeurs telles que consommation négative

In [16]:
conso_negative = df[df["consommation_kwh"] < 0]

print("Nombre de consommations négatives :", len(conso_negative))
conso_negative[["id_mesure", "date", "batiment", "consommation_kwh"]]

Nombre de consommations négatives : 4


,id_mesure,date,batiment,consommation_kwh
59,1246,2025-03-03 06:00:00,B5,-15.0
154,1046,2025-01-12 06:00:00,B2,-50.0
199,1146,2025-02-06 06:00:00,B7,-20.0
441,1346,2025-03-28 06:00:00,B5,-100.0


### f) Si une valeur est manifestement erronée et qu’on ne peut pas retrouver sa vraie valeur, la transformer en valeur manquante

In [17]:
df_avant = df.copy()   # copie de sécurité, pour comparer ou revenir en arrière

cols_a_corriger = ["temperature", "humidite", "occupation", "consommation_kwh"]
na_avant = df[cols_a_corriger].isna().sum()
print ("Nombre de valeurs manquantes avant correction :")
print(na_avant)

Nombre de valeurs manquantes avant correction :
temperature         12
humidite            11
occupation           6
consommation_kwh     5
dtype: int64


In [18]:
import numpy as np

### Convertion des valeurs erronnées

In [19]:
df.loc[(df["temperature"] < 0) | (df["temperature"] > 45), "temperature"] = np.nan
df.loc[(df["humidite"] < 0) | (df["humidite"] > 100), "humidite"] = np.nan
df.loc[df["occupation"] < 0, "occupation"] = np.nan
df.loc[df["consommation_kwh"] < 0, "consommation_kwh"] = np.nan

### Comparaison avant et après

In [20]:
na_apres = df[cols_a_corriger].isna().sum()

pd.DataFrame({
    "NaN avant": na_avant,
    "NaN après": na_apres,
    "convertis": na_apres - na_avant,
})

,NaN avant,NaN après,convertis
temperature,12,18,6
humidite,11,21,10
occupation,6,11,5
consommation_kwh,5,9,4


### g) rechercher des valeurs telles que catégories mal orthographiées

### Comparer le nombre de valeurs uniques avant et après nettoyage

In [22]:
cat_cols = ["batiment", "type_batiment", "zone", "mode_climatisation",
              "etat_systeme", "jour_semaine", "alerte"]
for c in cat_cols:
    brut = df[c].dropna().nunique()
    propre = df[c].dropna().str.strip().str.lower().nunique()
    print(f"{c:20} brutes : {brut:2}   après strip+lower : {propre:2}")

batiment             brutes :  8   après strip+lower :  8
type_batiment        brutes : 16   après strip+lower :  9
zone                 brutes :  4   après strip+lower :  4
mode_climatisation   brutes :  7   après strip+lower :  4
etat_systeme         brutes :  3   après strip+lower :  3
jour_semaine         brutes :  7   après strip+lower :  7
alerte               brutes :  2   après strip+lower :  2


### Examiner les colonnes où il y a des écarts

In [23]:
for c in ["type_batiment", "mode_climatisation"]:
    print(c)
    print("  brutes :", sorted(df[c].dropna().unique()))
    print("  propres:", sorted(df[c].dropna().str.strip().str.lower().unique()))
    print()

type_batiment
  brutes : [' Bureau ', ' UNIVERSITÉ', 'BUREAU', 'Bureau', 'Bureu', 'Centre commercial', 'Entrepôt', 'Hôpital', 'Université', 'bureau', 'centre commercial', 'ecole', 'entrepot', 'hôpital ', 'ÉCOLE', 'École']
  propres: ['bureau', 'bureu', 'centre commercial', 'ecole', 'entrepot', 'entrepôt', 'hôpital', 'université', 'école']

mode_climatisation
  brutes : ['BOOST', 'Boost', 'Eco', 'Normal', 'Normal ', 'normal', 'normale']
  propres: ['boost', 'eco', 'normal', 'normale']



### h) normaliser les catégories textuelles en supprimant les espaces puis en uniformisant la casse

In [24]:
# 1 et 2 : espaces puis casse
for c in cat_cols:
    df[c] = df[c].str.strip().str.lower()

# 3 : corriger les fautes (dictionnaire : colonne -> {faute : correction})
corrections = {
    "type_batiment": {
        "bureu": "bureau",
        "ecole": "école",
        "entrepot": "entrepôt",
    },
    "mode_climatisation": {
        "normale": "normal",
    },
}
df = df.replace(corrections)

# 4 : casse homogène (première lettre en majuscule)
for c in cat_cols:
    df[c] = df[c].str.capitalize()

### Vérification

In [25]:
for c in ["type_batiment", "mode_climatisation"]:
    print(c, df[c].nunique())
    print(df[c].value_counts(dropna=False), "\n")

type_batiment 6
type_batiment
Bureau               215
Centre commercial     66
École                 61
Hôpital               59
Entrepôt              53
Université            49
NaN                    4
Name: count, dtype: int64 

mode_climatisation 3
mode_climatisation
Normal    274
Eco       131
Boost      97
NaN         5
Name: count, dtype: int64 



### Contrôle qu'il ne reste aucun espace parasite

In [26]:
for c in cat_cols:
    propre = df[c].dropna()
    print(c, (propre != propre.str.strip()).sum())

batiment 0
type_batiment 0
zone 0
mode_climatisation 0
etat_systeme 0
jour_semaine 0
alerte 0
